# 🎧 Audiobook Studio MVP — Qwen3-TTS
**Purpose:** Long-form narration studio that converts any text/story into a narrated audiobook chapter.

## Quick Start Guide
1. Run setup and config cells
2. Define the reference voice
3. Add your chapter text
4. Generate and download!

In [ ]:
!pip install -q qwen-tts soundfile

import torch
import soundfile as sf
import os
import gc
import numpy as np
import urllib.request
from IPython.display import Audio, display


In [ ]:
MODEL_SIZE = "1.7B"
DTYPE = torch.bfloat16
OUTPUT_DIR = "/content/audiobook_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

def play_and_save(audio_data, sr, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    sf.write(path, audio_data, sr)
    print(f"Saved to {path}")
    display(Audio(path))


In [ ]:
from qwen_tts import Qwen3TTSModel

print("Loading base model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
base_model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=DTYPE,
    attn_implementation="sdpa"
)

print("Downloading reference audio...")
REF_AUDIO = "/content/clone.wav"
urllib.request.urlretrieve(
    "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav",
    REF_AUDIO
)
REF_TEXT = "Okay. Yeah. I resent you. I love you. I respect you. But you know what? You blew it! And thanks to you."

print("Creating voice clone prompt...")
narrator_prompt = base_model.create_voice_clone_prompt(ref_audio=REF_AUDIO, ref_text=REF_TEXT)
print("Narrator prompt created successfully.")


In [ ]:
USER_TEXT = """
The dense jungle Canopy parted just enough to let a single ray of sunlight through, illuminating the ancient stone ruin before them. Dr. Amelia Vance wiped the sweat from her brow, a triumphant smile breaking across her exhausted face.

For three weeks, the expedition had battled torrential rains, relentless mosquitoes, and paths that seemed to vanish into thin air. Many had urged her to turn back, convinced the Lost City of Xylos was nothing more than a myth meant to deter the greedy.

But Amelia knew better. Her grandfather's journals had been explicit, detailing the exact alignment of the stars and the winding river that led to this very spot. She stepped forward, her boots crunching softly on the overgrown foliage, heart pounding in her chest.

As she approached the monolithic entrance, strange carvings seemed to pulse with a faint, almost imperceptible blue light. It was just like the legends described—a place untouched by time, guarding secrets that could rewrite human history.

She reached out a trembling hand to touch the cold stone. In that moment, a low hum vibrated through the ground beneath her feet, and the heavy doors slowly began to slide open, revealing the dark, mysterious depths within.
"""

def chunk_text(text, max_chars=250):
    sentences = text.replace('\n', ' ').split('. ')
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence: continue
        if not sentence.endswith('.'): sentence += '.'
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += " " + sentence if current_chunk else sentence
        else:
            if current_chunk: chunks.append(current_chunk)
            current_chunk = sentence
    if current_chunk: chunks.append(current_chunk)
    return chunks

chunks = chunk_text(USER_TEXT)
print(f"Split text into {len(chunks)} chunks.")


In [ ]:
print("Generating chapter audio...")
all_audio = []
sr = 24000  # Default Qwen3-TTS sample rate

for i, chunk in enumerate(chunks):
    print(f"Generating chunk {i+1}/{len(chunks)}...")
    audio, _ = base_model.generate_voice_clone(
        text=chunk,
        language="English",
        voice_clone_prompt=narrator_prompt
    )
    all_audio.append(audio)
    # Add a brief pause between sentences
    pause = np.zeros(int(sr * 0.5), dtype=np.float32)  # 0.5s pause
    all_audio.append(pause)

combined_audio = np.concatenate(all_audio)
play_and_save(combined_audio, sr, "chapter_01.wav")


In [ ]:
import math
total_words = len(USER_TEXT.split())
total_duration_sec = len(combined_audio) / sr
wpm = (total_words / total_duration_sec) * 60
file_size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, "chapter_01.wav")) / (1024 * 1024)

print("--- Statistics ---")
print(f"Total Duration: {total_duration_sec:.2f} seconds")
print(f"File Size: {file_size_mb:.2f} MB")
print(f"Words Per Minute: {wpm:.1f} WPM")


In [ ]:
import shutil

try:
    from google.colab import files
    print("Zipping outputs...")
    shutil.make_archive("/content/audiobook_outputs", 'zip', OUTPUT_DIR)
    print("Downloading zip file...")
    files.download("/content/audiobook_outputs.zip")
except ImportError:
    print("google.colab import failed. Note: The download cell only works in Google Colab.")
